# Analyze a Document with Content Understanding

This notebook submits a publicly accessible document URL to the **invoiceAnalyzerDemo** analyzer created in `01_create_analyzer.ipynb`, then uses the `to_llm_input` helper from the **beta** `azure-ai-contentunderstanding` SDK to render the result in several LLM-ready formats.

> **Run `01_create_analyzer.ipynb` first** to ensure the analyzer exists.

## What you'll see
1. **Default** — fields + markdown (full LLM input)
2. **Fields only** — `include_markdown=False` (smallest token footprint)
3. **Markdown only** — `include_fields=False` (summarization / embedding scenarios)
4. **Custom metadata** — inject `source` (document URL) and `consumed_tokens` (pulled from `result.usage.total_tokens` when available) into the YAML front matter — useful for RAG pipelines that track provenance and cost

In [ ]:
%pip install azure-ai-contentunderstanding --pre --quiet
%pip install azure-identity python-dotenv --quiet

In [ ]:
import os
from azure.ai.contentunderstanding import ContentUnderstandingClient, to_llm_input
from azure.ai.contentunderstanding.models import AnalysisInput
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # loads .env from repo root

endpoint = os.environ["FOUNDRY_AI_SERVICES_ENDPOINT"].rstrip("/")
credential = DefaultAzureCredential()

ANALYZER_ID = "invoiceAnalyzerDemo"

client = ContentUnderstandingClient(endpoint=endpoint, credential=credential)
print("Client ready.")

In [ ]:
# Use a publicly accessible sample invoice PDF
# Replace with your own document URL if needed
DOCUMENT_URL = "https://github.com/Azure-Samples/azure-ai-content-understanding-python/raw/refs/heads/main/data/invoice.pdf"

print(f"Analyzing: {DOCUMENT_URL}\n")

poller = client.begin_analyze(
    analyzer_id=ANALYZER_ID,
    inputs=[AnalysisInput(url=DOCUMENT_URL)],
)
result = poller.result()
print("Analysis succeeded.")

## 1. Default — fields + markdown

`to_llm_input(result)` returns YAML front matter (content type, extracted fields, page numbers) followed by the markdown body. This is the most complete representation for an LLM.

In [ ]:
default_text = to_llm_input(result)
print(default_text)

## 2. Fields only

Set `include_markdown=False` to emit just the extracted fields. Useful for agentic workflows where the LLM only needs structured values — smallest token footprint.

In [ ]:
fields_only = to_llm_input(result, include_markdown=False)
print(fields_only)

## 3. Markdown only

Set `include_fields=False` when you only need the document text (e.g., summarization, classification, embedding generation).

In [ ]:
markdown_only = to_llm_input(result, include_fields=False)
print(markdown_only)

## 4. Custom metadata — `source` and `consumed_tokens`

Pass a `metadata` dict to inject custom key-value pairs into the YAML front matter. Useful for RAG pipelines that track provenance (the source document) and cost/usage (tokens consumed by the analyzer).

In [ ]:
# Token usage is exposed on the poller (poller.usage), not on the result.
# Example: {"tokens": {"gpt-4.1-input": 4787, "gpt-4.1-output": 479}, ...}
tokens = poller.usage.tokens

with_metadata = to_llm_input(
    result,
    include_markdown=False,  # skip the document body — keep only extracted fields
    metadata={
        "source": DOCUMENT_URL,
        "input_tokens": tokens["gpt-4.1-input"],
        "output_tokens": tokens["gpt-4.1-output"],
    },
)
print(with_metadata)